# 1.1 TensorFlow 與 Keras 基礎介紹

這份 Notebook 會快速走過 TensorFlow/Keras 的基本元件：tensor、variable、自動微分，以及第一個 Keras 模型訓練流程。


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
print('TensorFlow version:', tf.__version__)


## 1. Tensor 與 Variable

Tensor 是 TensorFlow 的基本資料單位；Variable 則是可被更新的參數。神經網路中的權重與偏差，就是訓練時會被更新的 variable。


In [ ]:
a = tf.constant([[1, 2], [3, 4]], dtype=tf.float32)
b = tf.constant([[10, 20], [30, 40]], dtype=tf.float32)
w = tf.Variable([[1.0], [2.0]])

print('a shape:', a.shape)
print('a dtype:', a.dtype)
print('a + b =')
print((a + b).numpy())
print('a @ w =')
print(tf.matmul(a, w).numpy())


## 2. Tensor 與 NumPy 互轉

實務上常用 pandas/NumPy 做資料整理，再轉交 TensorFlow/Keras 訓練模型。


In [ ]:
np_array = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
tensor = tf.convert_to_tensor(np_array)
back_to_numpy = tensor.numpy()

print(type(np_array), np_array.shape)
print(type(tensor), tensor.shape)
print(type(back_to_numpy), back_to_numpy.shape)


## 3. 自動微分 GradientTape

`tf.GradientTape()` 會記錄運算過程，並計算指定變數的梯度。這是神經網路能自動更新權重的基礎。


In [ ]:
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2 + 2 * x + 1

dy_dx = tape.gradient(y, x)
print('y =', y.numpy())
print('dy/dx =', dy_dx.numpy())


## 4. 第一個 Keras 模型：線性迴歸

接著用 Keras 建立一個最小的模型，學習 `y = 3x + 2` 這條線。


In [ ]:
rng = np.random.default_rng(42)
x = rng.uniform(-2, 2, size=(300, 1)).astype('float32')
y = (3 * x + 2 + rng.normal(0, 0.25, size=(300, 1))).astype('float32')

x_train, x_test = x[:240], x[240:]
y_train, y_test = y[:240], y[240:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05), loss='mse', metrics=['mae'])
history = model.fit(x_train, y_train, validation_split=0.2, epochs=80, verbose=0)

train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, test_result], index=['train', 'test'])


## 5. 預測新資料

訓練完成後，可以用 `model.predict()` 對新資料做推論。


In [ ]:
new_x = np.array([[-1.0], [0.0], [2.0]], dtype='float32')
pred = model.predict(new_x, verbose=0)

pd.DataFrame({
    'x': new_x.ravel(),
    'prediction': pred.ravel(),
    'expected_3x_plus_2': (3 * new_x + 2).ravel()
})


## 6. 觀察訓練曲線


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.title('Training Curve')
plt.show()


## 7. 小結

本 Notebook 完成了 TensorFlow/Keras 的最小完整流程：建立 tensor、理解 variable、自動微分、建立模型、訓練、評估與預測。
